# PocketLFM2 — train on Kaggle (LFM2 backbone + MTP)

**Before running:** right panel → *Session options* → **Accelerator = GPU T4 x2**, **Internet = On**.

Then **Run All**. It clones the repo, builds the LJSpeech latent cache (first run only),
trains `scripts2/train2.py`, and synthesizes a sample.

**Continue past the 12h limit:** *+ Add Input* → *Notebook Output* → add this notebook's previous version, then *Save & Run All*. It auto-resumes from the mounted `lfm2_last.pt`.

**To train on distilled data instead:** upload your `data/distill_cache` as a Kaggle Dataset, *+ Add Input* it, and it'll be picked up automatically.

In [ ]:
REPO_URL   = "https://github.com/emmudsouza/poclfmtts.git"
EPOCHS     = 200
BATCH_SIZE = 48      # 16GB T4; raise toward 64 if memory allows
GRAD_ACCUM = 1

import os
os.chdir("/kaggle/working")
if os.path.isdir("poclfmtts"):
    !cd poclfmtts && git pull
else:
    !git clone {REPO_URL}
%cd /kaggle/working/poclfmtts
!pip install -q -e . --no-deps && pip install -q pocket-tts
print("setup done")

In [ ]:
# Find a cache: extracted folder with index.txt, a .zip in inputs, or build LJSpeech once.
import os, glob, zipfile
def dirs_with_index(root): return [os.path.dirname(p) for p in glob.glob(root + '/**/index.txt', recursive=True)]

found = dirs_with_index('/kaggle/input')
if not found:
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        with zipfile.ZipFile(z) as zf: zf.extractall('/kaggle/working')
    found = dirs_with_index('/kaggle/working')

if found:
    CACHE = found[0]
    print('using uploaded cache:', CACHE)
else:
    CACHE = '/kaggle/working/ljspeech_cache'
    if not os.path.exists(CACHE + '/index.txt'):
        !python scripts/data_ljspeech.py --download --out {CACHE}
    print('built LJSpeech cache:', CACHE)


In [ ]:
# Resume from a previous session's checkpoint if one is mounted as input.
import glob, shutil, os
OUT = "/kaggle/working/runs/lfm2"
os.makedirs(OUT, exist_ok=True)
prev = glob.glob("/kaggle/input/*/lfm2_last.pt") + glob.glob("/kaggle/input/*/runs/lfm2/lfm2_last.pt")
resume = ""
if prev:
    resume = f"--resume {prev[0]}"
    print("resuming from", prev[0])

!python scripts2/train2.py --cache {CACHE} --out {OUT} --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --bf16 \
    --val-split 0.02 --workers 2 --save-every 1000 --log-every 50 {resume}

In [ ]:
# Synthesize a sample from the best checkpoint.
import os, IPython.display as ipd
best = "/kaggle/working/runs/lfm2/lfm2_best.pt"
if os.path.exists(best):
    !python scripts2/infer2.py --ckpt {best} --device cuda --num-steps 32 \
        --text "hello from the cloud, this is a test" --out /kaggle/working/sample.wav
    ipd.display(ipd.Audio("/kaggle/working/sample.wav"))
else:
    print("no checkpoint yet")

In [ ]:
# Slim inference-only checkpoint + download link.
import os, torch
from IPython.display import FileLink, display
best = "/kaggle/working/runs/lfm2/lfm2_best.pt"
if os.path.exists(best):
    ck = torch.load(best, map_location="cpu")
    slim = "/kaggle/working/lfm2_infer.pt"
    torch.save({"model": ck["model"], "cfg": ck["cfg"]}, slim)
    print(f"saved {slim} ({os.path.getsize(slim)/1e6:.0f} MB)")
    display(FileLink(slim))